In [4]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

DATA_PATH = "C:/Users/tirth/Desktop/ML project latest/transactions.csv"
MODEL_DIR = "model"
os.makedirs(MODEL_DIR, exist_ok=True)

CATEGORICAL_TARGET_ENCODE_COLS = [
    "transactionType",
    "merchantCategoryCode",
    "acqCountry",
    "posEntryMode",
    "cardPresent"
]

print("Loading data...")
df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)

if "isFraud" in df.columns:
    target_col = "isFraud"
elif "Class" in df.columns:
    target_col = "Class"
else:
    raise ValueError("No target column found (expected 'isFraud' or 'Class').")

for col in ["transactionDateTime", "accountOpenDate", "dateOfLastAddressChange"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

if "transactionDateTime" in df.columns:
    df["trans_hour"] = df["transactionDateTime"].dt.hour.fillna(12).astype(int)
    df["trans_dayofweek"] = df["transactionDateTime"].dt.dayofweek.fillna(3).astype(int)
else:
    df["trans_hour"] = 12
    df["trans_dayofweek"] = 3

if ("transactionDateTime" in df.columns) and ("accountOpenDate" in df.columns):
    df["days_since_account_open"] = (df["transactionDateTime"] - df["accountOpenDate"]).dt.days
else:
    df["days_since_account_open"] = np.nan

if ("transactionDateTime" in df.columns) and ("dateOfLastAddressChange" in df.columns):
    df["days_since_address_change"] = (df["transactionDateTime"] - df["dateOfLastAddressChange"]).dt.days
else:
    df["days_since_address_change"] = np.nan

for num_col in ["transactionAmount", "currentBalance", "creditLimit", "availableMoney"]:
    if num_col not in df.columns:
        df[num_col] = np.nan

drop_cols = []
for c in df.columns:
    if any(pat in c.lower() for pat in ["accountnumber", "cardlast4", "merchantname", "id", "email"]):
        drop_cols.append(c)
for dt_col in ["transactionDateTime", "accountOpenDate", "dateOfLastAddressChange"]:
    if dt_col in df.columns:
        drop_cols.append(dt_col)

drop_cols = list(set(drop_cols))
if drop_cols:
    df = df.drop(columns=drop_cols, errors="ignore")
    print("Dropped columns (examples):", drop_cols[:6])

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

encoders = {}
global_target_mean = df[target_col].mean()

for col in CATEGORICAL_TARGET_ENCODE_COLS:
    if col in df.columns:
        mapping = df.groupby(col)[target_col].mean().to_dict()
        encoders[col] = mapping
        df[col] = df[col].map(mapping)
    else:
        df[col] = global_target_mean
        encoders[col] = {}

df[CATEGORICAL_TARGET_ENCODE_COLS] = df[CATEGORICAL_TARGET_ENCODE_COLS].fillna(global_target_mean)

feature_columns = [
    "transactionAmount",
    "currentBalance",
    "creditLimit",
    "availableMoney",
    "days_since_account_open",
    "days_since_address_change",
    "trans_hour",
    "trans_dayofweek",
] + CATEGORICAL_TARGET_ENCODE_COLS

for c in feature_columns:
    if c not in df.columns:
        df[c] = np.nan

df[feature_columns] = df[feature_columns].fillna(df[feature_columns].median())

X = df[feature_columns].copy()
y = df[target_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training RandomForest...")
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=18,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)
clf.fit(X_train_scaled, y_train)

y_pred = clf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

joblib.dump(clf, os.path.join(MODEL_DIR, "fraud_model.joblib"))
joblib.dump(scaler, os.path.join(MODEL_DIR, "scaler.joblib"))
joblib.dump(encoders, os.path.join(MODEL_DIR, "encoders.joblib"))
joblib.dump(df[feature_columns].median().to_dict(), os.path.join(MODEL_DIR, "feature_means.joblib"))
joblib.dump(feature_columns, os.path.join(MODEL_DIR, "feature_columns.joblib"))

print("Saved model and preprocessors to", MODEL_DIR)


Loading data...
Initial shape: (786363, 22)
Dropped columns (examples): ['accountOpenDate', 'merchantName', 'cardLast4Digits', 'dateOfLastAddressChange', 'accountNumber', 'transactionDateTime']
Training RandomForest...
Accuracy: 0.8995441048368125
              precision    recall  f1-score   support

           0       0.99      0.91      0.95    154790
           1       0.06      0.34      0.10      2483

    accuracy                           0.90    157273
   macro avg       0.52      0.63      0.52    157273
weighted avg       0.97      0.90      0.93    157273

Saved model and preprocessors to model
